# Cymbal Pets analysis with BigQuery

This notebook checks the Cymbal Pets source tables, creates analytical views, explores customers and products, and produces business visualizations.

Complete the Cymbal Pets setup first and confirm that customers, orders, order_items, and products exist in the cymbal_pets dataset.

In [ ]:
%pip install -q google-cloud-bigquery db-dtypes pandas matplotlib

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('BIGQUERY_PROJECT')
if not PROJECT_ID:
    raise ValueError('Set GOOGLE_CLOUD_PROJECT or BIGQUERY_PROJECT.')

DATASET = 'cymbal_pets'
BQ = chr(96)
client = bigquery.Client(project=PROJECT_ID)
print(f'Using project: {PROJECT_ID}')

## Check the source tables

In [ ]:
query = f'''
SELECT table_name, table_type
FROM {BQ}{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.TABLES{BQ}
ORDER BY table_name
'''
tables_df = client.query(query).result().to_dataframe()
tables_df

required = {'customers', 'orders', 'order_items', 'products'}
missing = required - set(tables_df['table_name'])
if missing:
    raise RuntimeError(f'Missing tables: {sorted(missing)}')
print('All required source tables are available.')

## Create analytical views

In [ ]:
customer_view_sql = f'''
CREATE OR REPLACE VIEW {BQ}{PROJECT_ID}.{DATASET}.customer_analytics_vw{BQ} AS
SELECT
  c.customer_id,
  CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
  c.email, c.gender, c.address_city, c.address_state, c.loyalty_member,
  COUNT(DISTINCT o.order_id) AS total_orders,
  SUM(oi.quantity) AS total_items,
  SUM(oi.quantity * oi.price) AS total_revenue,
  ROUND(SAFE_DIVIDE(SUM(oi.quantity * oi.price), COUNT(DISTINCT o.order_id)), 2) AS avg_order_value,
  MIN(o.order_date) AS first_order_date,
  MAX(o.order_date) AS last_order_date
FROM {BQ}{PROJECT_ID}.{DATASET}.customers{BQ} c
LEFT JOIN {BQ}{PROJECT_ID}.{DATASET}.orders{BQ} o ON c.customer_id = o.customer_id
LEFT JOIN {BQ}{PROJECT_ID}.{DATASET}.order_items{BQ} oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, customer_name, c.email, c.gender, c.address_city, c.address_state, c.loyalty_member
'''
client.query(customer_view_sql).result()
print('Customer analytical view is ready.')

In [ ]:
product_view_sql = f'''
CREATE OR REPLACE VIEW {BQ}{PROJECT_ID}.{DATASET}.product_analytics_vw{BQ} AS
SELECT
  p.product_id, p.product_name, p.brand, p.category, p.subcategory, p.animal_type,
  p.price AS catalog_price, p.inventory_level, p.average_rating,
  COUNT(DISTINCT oi.order_id) AS total_orders,
  SUM(oi.quantity) AS units_sold,
  SUM(oi.quantity * oi.price) AS total_revenue,
  ROUND(SAFE_DIVIDE(SUM(oi.quantity * oi.price), NULLIF(SUM(oi.quantity), 0)), 2) AS avg_selling_price
FROM {BQ}{PROJECT_ID}.{DATASET}.products{BQ} p
LEFT JOIN {BQ}{PROJECT_ID}.{DATASET}.order_items{BQ} oi ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name, p.brand, p.category, p.subcategory, p.animal_type,
         p.price, p.inventory_level, p.average_rating
'''
client.query(product_view_sql).result()
print('Product analytical view is ready.')

## Customer analytics

In [ ]:
query = f'''
SELECT *
FROM {BQ}{PROJECT_ID}.{DATASET}.customer_analytics_vw{BQ}
ORDER BY total_revenue DESC
'''
customers_df = client.query(query).result().to_dataframe()
customers_df.head(10)

In [ ]:
customer_summary = pd.Series({
    'customers': customers_df['customer_id'].nunique(),
    'revenue': customers_df['total_revenue'].sum(),
    'orders': customers_df['total_orders'].sum(),
    'average_customer_value': customers_df['total_revenue'].mean(),
})
customer_summary

## Revenue by state

In [ ]:
query = f'''
SELECT address_state, SUM(total_revenue) AS revenue, SUM(total_orders) AS orders
FROM {BQ}{PROJECT_ID}.{DATASET}.customer_analytics_vw{BQ}
GROUP BY address_state
ORDER BY revenue DESC
'''
state_df = client.query(query).result().to_dataframe()
state_df.head(10)

In [ ]:
plot_df = state_df.head(15).sort_values('revenue')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df['address_state'], plot_df['revenue'], color='steelblue')
ax.set_title('Revenue by state')
ax.set_xlabel('Revenue')
plt.tight_layout()
plt.show()

## Product performance

In [ ]:
query = f'''
SELECT *
FROM {BQ}{PROJECT_ID}.{DATASET}.product_analytics_vw{BQ}
ORDER BY total_revenue DESC
'''
products_df = client.query(query).result().to_dataframe()
products_df.head(10)

In [ ]:
category_df = products_df.groupby('category', dropna=False).agg(
    revenue=('total_revenue', 'sum'),
    units_sold=('units_sold', 'sum'),
    products=('product_id', 'nunique'),
).sort_values('revenue', ascending=False)
category_df

In [ ]:
plot_df = category_df.sort_values('revenue')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df.index.astype(str), plot_df['revenue'], color='darkorange')
ax.set_title('Revenue by product category')
ax.set_xlabel('Revenue')
plt.tight_layout()
plt.show()

## Inventory risk

In [ ]:
inventory_risk_df = products_df[['product_name', 'category', 'inventory_level', 'units_sold', 'total_revenue']].loc[
    lambda df: (df['inventory_level'] <= 10) & (df['units_sold'] > 0)
].sort_values(['units_sold', 'inventory_level'], ascending=[False, True])
inventory_risk_df.head(20)

## Optional: write a customer segment table

In [ ]:
customer_segment_df = customers_df[['customer_id', 'customer_name', 'total_orders', 'total_revenue', 'avg_order_value']].copy()
customer_segment_df['segment'] = pd.cut(
    customer_segment_df['total_revenue'],
    bins=[-float('inf'), 100, 500, float('inf')],
    labels=['Low value', 'Standard value', 'High value'],
)
customer_segment_df.head()

In [ ]:
WRITE_RESULTS = False
DESTINATION = f'{PROJECT_ID}.{DATASET}.customer_segments_notebook'
if WRITE_RESULTS:
    load_job = client.load_table_from_dataframe(customer_segment_df, DESTINATION, job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE'))
    load_job.result()
    print(f'Wrote {len(customer_segment_df):,} rows to {DESTINATION}')
else:
    print('Result writing is disabled. Set WRITE_RESULTS = True to write the table.')

## Next steps

- Connect the analytical views to the Cymbal Pets Looker Studio dashboard.
- Add a monthly revenue query and visualize the trend.
- Compare notebook results with business-queries.sql.
- Ask Gemini to generate one of the queries and validate its joins and aggregations.